# Sigiriya SinLlama generation and verification diagnostic v1
Select a GPU runtime in Colab. Upload the diagnostic ZIP built locally with
`python -B -m scripts.sinllama_diagnostic_v1 bundle`. The ZIP contains the saved evidence packet and a small runtime.
The model downloads from Hugging Face on first load.


In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

import torch
from google.colab import files

assert torch.cuda.is_available(), "Select a GPU runtime before running the experiment."
bundle_name = "sigiriya_sinllama_diagnostic_v1.zip"
uploaded = files.upload()
assert bundle_name in uploaded, f"Upload {bundle_name}"
workspace = Path(mkdtemp(prefix="sigiriya_diag_v1_", dir="/content"))
with ZipFile(bundle_name) as archive:
    archive.extractall(workspace)
print(torch.cuda.get_device_name())


In [ ]:
import subprocess
import sys

output = workspace / "results" / "sinllama_diagnostic_v1"
for mode in ("offline", "gpu"):
    subprocess.run(
        [sys.executable, "-B", "-m", "scripts.sinllama_diagnostic_v1", mode],
        cwd=workspace, check=True,
    )


In [ ]:
import json
from collections import Counter

records = [json.loads(line) for line in
           (output / "generation_v1.jsonl").read_text(encoding="utf-8").splitlines()]
assert len(records) == 9, "Expected three profiles for each of three questions."
for row in records:
    print(row["question_id"], row["profile"], row["generation_findings"])
    if row["strict"] is None:
        print(row["generation_error"])
    else:
        print("  strict", row["strict"]["reason"])
        print("  experimental", row["experimental"]["reason"])
errors = Counter(row["generation_error"] for row in records if "generation_error" in row)
print("Successful inference rows:", len(records) - sum(errors.values()), "/ 9")


In [ ]:
from shutil import make_archive

archive_path = make_archive(str(workspace / "sinllama_diagnostic_v1_outputs"), "zip", output)
files.download(archive_path)
